In [2]:
from gurobipy import GRB, Model, quicksum
import gurobipy as gb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

In [3]:
costs = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/costs.csv')
randomness = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/randomness.csv')

In [4]:
costs

,Unnamed: 0,Station_0,Station_1,Station_2,Station_3,Station_4,Station_5,Station_6,Station_7,Station_8,Station_9,Station_10,Station_11,Station_12,Station_13,Station_14
0,Station_0,0,43,7,27,20,22,43,28,42,30,22,27,36,35,16
1,Station_1,22,0,12,38,19,14,46,13,30,47,18,29,11,32,9
2,Station_2,23,46,0,14,43,14,33,23,13,45,12,12,28,16,23
3,Station_3,10,45,14,0,44,35,29,25,5,29,46,46,25,34,21
4,Station_4,39,32,15,44,0,24,48,47,25,34,14,47,45,10,23
5,Station_5,27,14,44,23,8,0,24,39,16,40,13,22,19,17,11
6,Station_6,49,10,27,25,40,16,0,10,46,26,30,15,47,19,10
7,Station_7,27,18,32,8,37,40,29,0,42,19,41,20,40,9,23
8,Station_8,14,37,8,14,44,43,23,5,0,33,47,12,39,33,26
9,Station_9,27,5,27,48,47,6,39,22,17,0,40,42,15,22,23


In [5]:
randomness

,Unnamed: 0,Probability,Mean_Demand,Std_Dev_Demand
0,Station_1,0.549406,184,29
1,Station_2,0.597865,310,33
2,Station_3,0.274362,465,14
3,Station_4,0.204316,193,39
4,Station_5,0.241876,118,43
5,Station_6,0.703453,265,33
6,Station_7,0.699967,169,13
7,Station_8,0.780134,273,46
8,Station_9,0.823935,393,43
9,Station_10,0.709039,308,33


In [6]:
# # Print original structure
# print("Original DataFrame structures:")
# print("\nCosts DataFrame columns:")
# print(costs.columns)
# print("\nRandomness DataFrame columns:")
# print(randomness.columns)

# # Print first few rows of each
# print("\nCosts DataFrame head:")
# print(costs.head())
# print("\nRandomness DataFrame head:")
# print(randomness.head())

In [7]:
# def generate_scenario():
#     stations_needing_fuel = []
#     demands = {}
    
#     for station in randomness.index:  # Will iterate through Station_1 to Station_14
#         if random.random() < randomness.loc[station, 'Probability']:
#             stations_needing_fuel.append(station)
#             demand = np.random.normal(
#                 randomness.loc[station, 'Mean_Demand'],
#                 randomness.loc[station, 'Std_Dev_Demand']
#             )
#             demands[station] = float(max(0, demand))
    
#     return stations_needing_fuel, demands

# def solve_routing_problem(stations, demands, fixed_truck_size=None):
#     model = Model("FuelFlow")
#     model.Params.OutputFlag = 0
    
#     if not stations:
#         return 0
    
#     # Add storage facility to route
#     all_nodes = ['Station_0'] + stations
#     n = len(all_nodes)
    
#     # Decision variables for routes
#     x = {}
#     for i in all_nodes:
#         for j in all_nodes:
#             if i != j:
#                 x[i,j] = model.addVar(vtype=GRB.BINARY, name=f'route_{i}_{j}')
    
#     # Truck size variable
#     if fixed_truck_size is None:
#         truck_size = model.addVar(name='truck_size')
#     else:
#         truck_size = fixed_truck_size
    
#     # Objective function
#     travel_cost = quicksum(float(costs.loc[i,j]) * x[i,j] 
#                           for i in all_nodes for j in all_nodes if i != j)
    
#     total_demand = float(sum(demands.values()))
#     oversized_penalty = 0.09 * (truck_size - total_demand)
    
#     # Handle undersized penalty using a variable
#     shortage = model.addVar(name='shortage')
#     undersized_penalty = 0.13 * shortage
#     model.addConstr(shortage >= total_demand - truck_size)
    
#     model.setObjective(travel_cost + oversized_penalty + undersized_penalty, GRB.MINIMIZE)
    
#     # Constraints
#     # Start at storage facility
#     model.addConstr(quicksum(x['Station_0',j] for j in stations) == 1)
    
#     # Flow conservation
#     for k in stations:
#         model.addConstr(
#             quicksum(x[i,k] for i in all_nodes if i != k) == 
#             quicksum(x[k,j] for j in all_nodes if j != k)
#         )
    
#     # Visit each station once
#     for j in stations:
#         model.addConstr(quicksum(x[i,j] for i in all_nodes if i != j) == 1)
    
#     # Return to storage facility
#     model.addConstr(quicksum(x[i,'Station_0'] for i in stations) == 1)
    
#     # Subtour elimination
#     if len(stations) > 1:
#         u = {}
#         for i in stations:
#             u[i] = model.addVar()
        
#         for i in stations:
#             for j in stations:
#                 if i != j:
#                     model.addConstr(u[i] - u[j] + n*x[i,j] <= n-1)
    
#     model.optimize()
    
#     if model.status == GRB.OPTIMAL:
#         return model.objVal
#     return float('inf')

# # Run Monte Carlo simulation with SAA
# total_cost = 0
# num_trials = 20
# scenarios_per_trial = 10

# for trial in range(num_trials):
#     trial_cost = 0
    
#     for scenario in range(scenarios_per_trial):
#         stations, demands = generate_scenario()
#         scenario_cost = solve_routing_problem(stations, demands)
#         trial_cost += scenario_cost
    
#     total_cost += trial_cost / scenarios_per_trial

# expected_cost = total_cost / num_trials
# print(f"Optimal expected cost: ${expected_cost:.2f}")

In [8]:

# def generate_scenario():
#     stations_needing_fuel = []
#     demands = {}
    
#     for station in randomness.index:
#         if random.random() < randomness.loc[station, 'Probability']:
#             stations_needing_fuel.append(station)
#             demand = np.random.normal(
#                 randomness.loc[station, 'Mean_Demand'],
#                 randomness.loc[station, 'Std_Dev_Demand']
#             )
#             demands[station] = float(max(0, demand))
    
#     return stations_needing_fuel, demands

# def solve_routing_problem(stations, demands, fixed_truck_size=None):
#     model = Model("FuelFlow")
#     model.Params.OutputFlag = 0
    
#     if not stations:
#         return 0.0  # Return 0.0 instead of 0
    
#     # Add storage facility to route
#     all_nodes = ['Station_0'] + stations
#     n = len(all_nodes)
    
#     # Decision variables for routes
#     x = {}
#     for i in all_nodes:
#         for j in all_nodes:
#             if i != j:
#                 x[i,j] = model.addVar(vtype=GRB.BINARY, name=f'route_{i}_{j}')
    
#     # Truck size variable
#     if fixed_truck_size is None:
#         truck_size = model.addVar(name='truck_size')
#     else:
#         truck_size = fixed_truck_size
    
#     # Objective function
#     travel_cost = quicksum(float(costs.loc[i,j]) * x[i,j] 
#                           for i in all_nodes for j in all_nodes if i != j)
    
#     total_demand = float(sum(demands.values()))
#     oversized_penalty = 0.09 * (truck_size - total_demand)
    
#     # Handle undersized penalty using a variable
#     shortage = model.addVar(name='shortage')
#     undersized_penalty = 0.13 * shortage
#     model.addConstr(shortage >= total_demand - truck_size)
    
#     model.setObjective(travel_cost + oversized_penalty + undersized_penalty, GRB.MINIMIZE)
    
#     # Constraints
#     # Start at storage facility
#     model.addConstr(quicksum(x['Station_0',j] for j in stations) == 1)
    
#     # Flow conservation
#     for k in stations:
#         model.addConstr(
#             quicksum(x[i,k] for i in all_nodes if i != k) == 
#             quicksum(x[k,j] for j in all_nodes if j != k)
#         )
    
#     # Visit each station once
#     for j in stations:
#         model.addConstr(quicksum(x[i,j] for i in all_nodes if i != j) == 1)
    
#     # Return to storage facility
#     model.addConstr(quicksum(x[i,'Station_0'] for i in stations) == 1)
    
#     # Subtour elimination
#     if len(stations) > 1:
#         u = {}
#         for i in stations:
#             u[i] = model.addVar()
        
#         for i in stations:
#             for j in stations:
#                 if i != j:
#                     model.addConstr(u[i] - u[j] + n*x[i,j] <= n-1)
    
#     model.optimize()
    
#     if model.status == GRB.OPTIMAL:
#         return float(model.objVal)  # Ensure we return a float
#     return float('inf')  # Return infinity instead of None

# # Calculate Wait-and-See solution
# ws_total = 0.0  # Initialize as float
# num_trials = 20
# scenarios_per_trial = 10

# for trial in range(num_trials):
#     trial_ws = 0.0  # Initialize as float
    
#     for scenario in range(scenarios_per_trial):
#         stations, demands = generate_scenario()
#         scenario_cost = solve_routing_problem(stations, demands)
#         if scenario_cost != float('inf'):  # Only add if solution found
#             trial_ws += scenario_cost
    
#     ws_total += trial_ws / scenarios_per_trial

# ws_value = ws_total / num_trials

# # Calculate Recourse Problem solution
# rp_total = 0.0  # Initialize as float

# for trial in range(num_trials):
#     trial_cost = 0.0  # Initialize as float
    
#     for scenario in range(scenarios_per_trial):
#         stations, demands = generate_scenario()
#         scenario_cost = solve_routing_problem(stations, demands)
#         if scenario_cost != float('inf'):  # Only add if solution found
#             trial_cost += scenario_cost
    
#     rp_total += trial_cost / scenarios_per_trial

# rp_value = rp_total / num_trials

# # Calculate EVPI
# evpi = rp_value - ws_value

# print(f"Wait-and-See (WS) Value: ${ws_value:.2f}")
# print(f"Recourse Problem (RP) Value: ${rp_value:.2f}")
# print(f"Expected Value of Perfect Information (EVPI): ${evpi:.2f}")

In [9]:
# def solve_routing_problem(stations, demands, fixed_truck_size=None):
#     model = Model("FuelFlow")
#     model.Params.OutputFlag = 0
    
#     if not stations:
#         return 0.0
    
#     # Add storage facility to route
#     all_nodes = ['Station_0'] + stations
#     n = len(all_nodes)
    
#     # Decision variables for routes
#     x = {}
#     for i in all_nodes:
#         for j in all_nodes:
#             if i != j:
#                 x[i,j] = model.addVar(vtype=GRB.BINARY)
    
#     # Truck size variable
#     if fixed_truck_size is None:
#         truck_size = model.addVar()
#     else:
#         truck_size = fixed_truck_size
    
#     # Objective function
#     travel_cost = quicksum(float(costs.loc[i,j]) * x[i,j] 
#                           for i in all_nodes for j in all_nodes if i != j)
    
#     total_demand = float(sum(demands.values()))
#     oversized_penalty = 0.09 * (truck_size - total_demand)
#     shortage = model.addVar()
#     undersized_penalty = 0.13 * shortage
    
#     model.addConstr(shortage >= total_demand - truck_size)
#     model.setObjective(travel_cost + oversized_penalty + undersized_penalty, GRB.MINIMIZE)
    
#     # Add routing constraints
#     model.addConstr(quicksum(x['Station_0',j] for j in stations) == 1)
#     for k in stations:
#         model.addConstr(
#             quicksum(x[i,k] for i in all_nodes if i != k) == 
#             quicksum(x[k,j] for j in all_nodes if j != k)
#         )
#     for j in stations:
#         model.addConstr(quicksum(x[i,j] for i in all_nodes if i != j) == 1)
#     model.addConstr(quicksum(x[i,'Station_0'] for i in stations) == 1)
    
#     model.optimize()
#     return model.objVal if model.status == GRB.OPTIMAL else float('inf')

# def solve_ev_problem():
#     # Solve using mean demands for all stations
#     mean_demands = {station: randomness.loc[station, 'Mean_Demand'] 
#                    for station in randomness.index}
#     return solve_routing_problem(list(randomness.index), mean_demands)

# def generate_scenario():
#     stations = []
#     demands = {}
#     for station in randomness.index:
#         if random.random() < randomness.loc[station, 'Probability']:
#             stations.append(station)
#             demands[station] = max(0, np.random.normal(
#                 randomness.loc[station, 'Mean_Demand'],
#                 randomness.loc[station, 'Std_Dev_Demand']
#             ))
#     return stations, demands

# # Calculate EV, EEV, and VSS
# ev_solution = solve_ev_problem()
# eev_total = 0
# rp_total = 0

# for _ in range(20):  # 20 trials
#     trial_eev = 0
#     trial_rp = 0
    
#     for _ in range(10):  # 10 scenarios per trial
#         stations, demands = generate_scenario()
#         # EEV: Use EV solution's truck size
#         eev_cost = solve_routing_problem(stations, demands, ev_solution)
#         # RP: Solve without fixed truck size
#         rp_cost = solve_routing_problem(stations, demands)
        
#         trial_eev += eev_cost
#         trial_rp += rp_cost
    
#     eev_total += trial_eev / 10
#     rp_total += trial_rp / 10

# eev = eev_total / 20
# rp = rp_total / 20
# vss = eev - rp

# print(f"EEV: ${eev:.2f}")
# print(f"RP: ${rp:.2f}")
# print(f"VSS: ${vss:.2f}")

Q.) h

VSS (Value of Stochastic Solution) = $89.16
This means FuelFlow would spend $89.16 more per day by using average demands (deterministic approach) instead of considering uncertainty (stochastic approach)
Annually, this represents potential savings of approximately $32,543 (89.16 × 365 days)
Managerial insight: It's worth investing in stochastic programming methods for planning rather than using simple averages
Comparing EEV ($198.24) vs RP ($109.08):
Using average demands (EEV approach) costs $198.24 per day
Using stochastic programming (RP approach) costs $109.08 per day
This significant difference (44.9% reduction) shows that:
Considering uncertainty in daily planning is crucial
Simple average-based planning is inadequate for this problem
The investment in more sophisticated planning methods is justified
Managerial Recommendations:
Implement stochastic programming for daily routing and truck size decisions
Don't rely on average demand forecasts alone
The potential savings ($89.16 per day) justify investing in better planning systems

In [10]:
# """
# Question (e): Create a stochastic program using a Monte Carlo simulation of 20 trials 
# with 10 scenarios per trial (i.e., use SAA). What is the optimal expected cost?
# """
# def generate_scenario():
#     """Generate a random scenario based on station probabilities and demands"""
#     stations = []
#     demands = {}
#     for station in randomness.index:
#         if random.random() < randomness.loc[station, 'Probability']:
#             stations.append(station)
#             demands[station] = max(0, np.random.normal(
#                 randomness.loc[station, 'Mean_Demand'],
#                 randomness.loc[station, 'Std_Dev_Demand']
#             ))
#     return stations, demands

# def solve_saa_problem(num_trials=20, scenarios_per_trial=10):
#     """Solve the stochastic program using Sample Average Approximation"""
#     trial_solutions = []
    
#     for trial in range(num_trials):
#         # Generate scenarios for this trial
#         scenarios = [generate_scenario() for _ in range(scenarios_per_trial)]
        
#         # Create model for this trial
#         model = Model("SAA_Trial")
#         model.Params.OutputFlag = 0
        
#         # First-stage decision variable (truck size)
#         y = model.addVar(name='truck_size')
        
#         # Second-stage variables
#         x = {}  # routing decisions
#         shortage = {}  # shortage amounts
        
#         # Total objective for all scenarios
#         total_obj = 0
        
#         # For each scenario
#         for k, (stations, demands) in enumerate(scenarios):
#             all_nodes = ['Station_0'] + stations
            
#             # Routing variables for this scenario
#             for i in all_nodes:
#                 for j in all_nodes:
#                     if i != j:
#                         x[k,i,j] = model.addVar(vtype=GRB.BINARY)
            
#             shortage[k] = model.addVar()
            
#             # Constraints
#             # Must start from depot
#             model.addConstr(quicksum(x[k,'Station_0',j] for j in stations) == 1)
            
#             # Flow conservation
#             for node in stations:
#                 model.addConstr(
#                     quicksum(x[k,i,node] for i in all_nodes if i != node) ==
#                     quicksum(x[k,node,j] for j in all_nodes if j != node)
#                 )
            
#             # Visit each station once
#             for j in stations:
#                 model.addConstr(quicksum(x[k,i,j] for i in all_nodes if i != j) == 1)
            
#             # Return to depot
#             model.addConstr(quicksum(x[k,i,'Station_0'] for i in stations) == 1)
            
#             # Define shortage
#             total_demand = sum(demands.values())
#             model.addConstr(shortage[k] >= total_demand - y)
            
#             # Add to objective
#             scenario_obj = (
#                 quicksum(costs.loc[i,j] * x[k,i,j] 
#                         for i in all_nodes for j in all_nodes if i != j) +
#                 0.09 * (y - total_demand) +
#                 0.13 * shortage[k]
#             )
#             total_obj += scenario_obj
        
#         # Set objective (average over scenarios)
#         model.setObjective(total_obj / scenarios_per_trial, GRB.MINIMIZE)
        
#         # Optimize
#         model.optimize()
        
#         if model.status == GRB.OPTIMAL:
#             trial_solutions.append(model.objVal)
    
#     # Calculate average optimal cost across all trials
#     average_cost = np.mean(trial_solutions)
#     std_dev = np.std(trial_solutions)
    
#     return average_cost, std_dev

# # Solve and print results
# optimal_cost, std_dev = solve_saa_problem()
# print(f"Optimal Expected Cost: ${optimal_cost:.2f}")
# print(f"Standard Deviation: ${std_dev:.2f}")
# print(f"95% Confidence Interval: [${optimal_cost - 1.96*std_dev/np.sqrt(20):.2f}, "
#       f"${optimal_cost + 1.96*std_dev/np.sqrt(20):.2f}]")

In [11]:
# """
# Question (e): Create a stochastic program using a Monte Carlo simulation of 20 trials 
# with 10 scenarios per trial (i.e., use SAA). What is the optimal expected cost?
# """
# def generate_scenario():
#     stations = []
#     demands = {}
#     for station in randomness.index:
#         if random.random() < randomness.loc[station, 'Probability']:
#             stations.append(station)
#             demands[station] = max(0, np.random.normal(
#                 randomness.loc[station, 'Mean_Demand'],
#                 randomness.loc[station, 'Std_Dev_Demand']
#             ))
#     return stations, demands

# def solve_recourse_problem():
#     total_cost = 0
#     num_trials = 20
#     scenarios_per_trial = 10
    
#     for trial in range(num_trials):
#         # Create model for this trial
#         model = Model("RP_Trial")
#         model.Params.OutputFlag = 0
        
#         # Generate scenarios for this trial
#         scenarios = [generate_scenario() for _ in range(scenarios_per_trial)]
        
#         # First-stage variable (truck size)
#         y = model.addVar(name='truck_size')
        
#         # Second-stage variables for each scenario
#         x = {}
#         shortage = {}
#         obj_total = 0
        
#         # For each scenario in this trial
#         for k, (stations, demands) in enumerate(scenarios):
#             all_nodes = ['Station_0'] + stations
            
#             # Create routing variables for this scenario
#             for i in all_nodes:
#                 for j in all_nodes:
#                     if i != j:
#                         x[k,i,j] = model.addVar(vtype=GRB.BINARY)
            
#             shortage[k] = model.addVar()
            
#             # Constraints for this scenario
#             # Start at depot
#             model.addConstr(quicksum(x[k,'Station_0',j] for j in stations) == 1)
            
#             # Flow conservation
#             for node in stations:
#                 model.addConstr(
#                     quicksum(x[k,i,node] for i in all_nodes if i != node) ==
#                     quicksum(x[k,node,j] for j in all_nodes if j != node)
#                 )
            
#             # Visit each station once
#             for j in stations:
#                 model.addConstr(quicksum(x[k,i,j] for i in all_nodes if i != j) == 1)
            
#             # Return to depot
#             model.addConstr(quicksum(x[k,i,'Station_0'] for i in stations) == 1)
            
#             # Shortage definition
#             total_demand = sum(demands.values())
#             model.addConstr(shortage[k] >= total_demand - y)
            
#             # Add to objective
#             obj_total += (quicksum(costs.loc[i,j] * x[k,i,j] 
#                                  for i in all_nodes for j in all_nodes if i != j)
#                          + 0.09 * (y - total_demand)
#                          + 0.13 * shortage[k])
        
#         # Set objective for this trial (average over scenarios)
#         model.setObjective(obj_total / scenarios_per_trial, GRB.MINIMIZE)
#         model.optimize()
        
#         if model.status == GRB.OPTIMAL:
#             total_cost += model.objVal
    
#     # Return average cost over all trials
#     return total_cost / num_trials

# # Calculate and print RP value
# rp_value = solve_recourse_problem()
# print(f"Recourse Problem (RP) Value: ${rp_value:.2f}")

In [12]:
# """
# Question (f): Using a Monte Carlo simulation of 20 trials with 10 scenarios per trial, 
# what is the expected value of reacting with perfect foresight (WS) and the expected 
# value of perfect information (EVPI) associated with this stochastic program?
# """
# def generate_scenario():
#     stations = []
#     demands = {}
#     for station in randomness.index:
#         if random.random() < randomness.loc[station, 'Probability']:
#             stations.append(station)
#             demands[station] = max(0, np.random.normal(
#                 randomness.loc[station, 'Mean_Demand'],
#                 randomness.loc[station, 'Std_Dev_Demand']
#             ))
#     return stations, demands

# def solve_wait_and_see():
#     total_cost = 0
#     num_trials = 20
#     scenarios_per_trial = 10
    
#     for trial in range(num_trials):
#         trial_cost = 0
#         for scenario in range(scenarios_per_trial):
#             stations, demands = generate_scenario()
            
#             model = Model("WS_Scenario")
#             model.Params.OutputFlag = 0
            
#             # Variables
#             y = model.addVar(name='truck_size')
#             x = {}
#             all_nodes = ['Station_0'] + stations
            
#             for i in all_nodes:
#                 for j in all_nodes:
#                     if i != j:
#                         x[i,j] = model.addVar(vtype=GRB.BINARY)
            
#             shortage = model.addVar()
            
#             # Constraints
#             model.addConstr(quicksum(x['Station_0',j] for j in stations) == 1)
            
#             for node in stations:
#                 model.addConstr(
#                     quicksum(x[i,node] for i in all_nodes if i != node) ==
#                     quicksum(x[node,j] for j in all_nodes if j != node)
#                 )
            
#             for j in stations:
#                 model.addConstr(quicksum(x[i,j] for i in all_nodes if i != j) == 1)
            
#             model.addConstr(quicksum(x[i,'Station_0'] for i in stations) == 1)
            
#             total_demand = sum(demands.values())
#             model.addConstr(shortage >= total_demand - y)
            
#             obj = (quicksum(costs.loc[i,j] * x[i,j] 
#                           for i in all_nodes for j in all_nodes if i != j)
#                   + 0.09 * (y - total_demand)
#                   + 0.13 * shortage)
            
#             model.setObjective(obj, GRB.MINIMIZE)
#             model.optimize()
            
#             if model.status == GRB.OPTIMAL:
#                 trial_cost += model.objVal
            
#         total_cost += trial_cost / scenarios_per_trial
    
#     return total_cost / num_trials

# def solve_rp():
#     # Similar to wait-and-see but with same truck size across scenarios
#     model = Model("RP")
#     model.Params.OutputFlag = 0
    
#     # Generate all scenarios upfront
#     scenarios = [generate_scenario() for _ in range(10)]
    
#     # First-stage variable (same truck size for all scenarios)
#     y = model.addVar(name='truck_size')
    
#     # Second-stage variables for each scenario
#     x = {}
#     shortage = {}
#     obj_total = 0
    
#     for k, (stations, demands) in enumerate(scenarios):
#         all_nodes = ['Station_0'] + stations
        
#         # Variables for this scenario
#         for i in all_nodes:
#             for j in all_nodes:
#                 if i != j:
#                     x[k,i,j] = model.addVar(vtype=GRB.BINARY)
#         shortage[k] = model.addVar()
        
#         # Constraints for this scenario
#         model.addConstr(quicksum(x[k,'Station_0',j] for j in stations) == 1)
        
#         for node in stations:
#             model.addConstr(
#                 quicksum(x[k,i,node] for i in all_nodes if i != node) ==
#                 quicksum(x[k,node,j] for j in all_nodes if j != node)
#             )
        
#         for j in stations:
#             model.addConstr(quicksum(x[k,i,j] for i in all_nodes if i != j) == 1)
        
#         model.addConstr(quicksum(x[k,i,'Station_0'] for i in stations) == 1)
        
#         total_demand = sum(demands.values())
#         model.addConstr(shortage[k] >= total_demand - y)
        
#         # Add to objective
#         obj_total += (quicksum(costs.loc[i,j] * x[k,i,j] 
#                              for i in all_nodes for j in all_nodes if i != j)
#                      + 0.09 * (y - total_demand)
#                      + 0.13 * shortage[k])
    
#     model.setObjective(obj_total / len(scenarios), GRB.MINIMIZE)
#     model.optimize()
    
#     return model.objVal

# # Calculate WS and EVPI
# ws_value = solve_wait_and_see()
# rp_value = solve_rp()
# evpi = rp_value - ws_value

# print(f"Wait-and-See (WS) Value: ${ws_value:.2f}")
# print(f"Recourse Problem (RP) Value: ${rp_value:.2f}")
# print(f"Expected Value of Perfect Information (EVPI): ${evpi:.2f}")

In [13]:
# def generate_scenario():
#     """Generate a random scenario based on station probabilities and demands"""
#     stations = []
#     demands = {}
#     for station in randomness.index:
#         if random.random() < randomness.loc[station, 'Probability']:
#             stations.append(station)
#             demands[station] = max(0, np.random.normal(
#                 randomness.loc[station, 'Mean_Demand'],
#                 randomness.loc[station, 'Std_Dev_Demand']
#             ))
#     return stations, demands

# def solve_rp():
#     """Solve the recourse problem using SAA"""
#     total_cost = 0
#     num_trials = 20
#     scenarios_per_trial = 10
    
#     for trial in range(num_trials):
#         # Create model for this trial
#         model = Model("RP_Trial")
#         model.Params.OutputFlag = 0
        
#         # Generate scenarios for this trial
#         scenarios = [generate_scenario() for _ in range(scenarios_per_trial)]
        
#         # First-stage variable (truck size)
#         y = model.addVar(name='truck_size')
        
#         # Second-stage variables
#         x = {}
#         shortage = {}
#         obj_total = 0
        
#         # For each scenario
#         for k, (stations, demands) in enumerate(scenarios):
#             all_nodes = ['Station_0'] + stations
            
#             # Routing variables for this scenario
#             for i in all_nodes:
#                 for j in all_nodes:
#                     if i != j:
#                         x[k,i,j] = model.addVar(vtype=GRB.BINARY)
            
#             shortage[k] = model.addVar()
            
#             # Constraints
#             model.addConstr(quicksum(x[k,'Station_0',j] for j in stations) == 1)
            
#             for node in stations:
#                 model.addConstr(
#                     quicksum(x[k,i,node] for i in all_nodes if i != node) ==
#                     quicksum(x[k,node,j] for j in all_nodes if j != node)
#                 )
            
#             for j in stations:
#                 model.addConstr(quicksum(x[k,i,j] for i in all_nodes if i != j) == 1)
            
#             model.addConstr(quicksum(x[k,i,'Station_0'] for i in stations) == 1)
            
#             total_demand = sum(demands.values())
#             model.addConstr(shortage[k] >= total_demand - y)
            
#             scenario_obj = (
#                 quicksum(costs.loc[i,j] * x[k,i,j] 
#                         for i in all_nodes for j in all_nodes if i != j) +
#                 0.09 * (y - total_demand) +
#                 0.13 * shortage[k]
#             )
#             obj_total += scenario_obj
        
#         model.setObjective(obj_total / scenarios_per_trial, GRB.MINIMIZE)
#         model.optimize()
        
#         if model.status == GRB.OPTIMAL:
#             total_cost += model.objVal
    
#     return total_cost / num_trials

# def solve_mean_value_problem():
#     """Solve the mean value problem where all stations are visited with mean demands"""
#     model = Model("MeanValue")
#     model.Params.OutputFlag = 0
    
#     # All stations are visited
#     stations = list(randomness.index)
#     all_nodes = ['Station_0'] + stations
    
#     # Variables
#     y = model.addVar(name='truck_size')
#     x = {}
#     for i in all_nodes:
#         for j in all_nodes:
#             if i != j:
#                 x[i,j] = model.addVar(vtype=GRB.BINARY)
    
#     shortage = model.addVar()
    
#     # Constraints
#     model.addConstr(quicksum(x['Station_0',j] for j in stations) == 1)
    
#     for node in stations:
#         model.addConstr(
#             quicksum(x[i,node] for i in all_nodes if i != node) ==
#             quicksum(x[node,j] for j in all_nodes if j != node)
#         )
    
#     for j in stations:
#         model.addConstr(quicksum(x[i,j] for i in all_nodes if i != j) == 1)
    
#     model.addConstr(quicksum(x[i,'Station_0'] for i in stations) == 1)
    
#     total_mean_demand = sum(randomness['Mean_Demand'])
#     model.addConstr(shortage >= total_mean_demand - y)
    
#     obj = (quicksum(costs.loc[i,j] * x[i,j] 
#                    for i in all_nodes for j in all_nodes if i != j)
#            + 0.09 * (y - total_mean_demand)
#            + 0.13 * shortage)
    
#     model.setObjective(obj, GRB.MINIMIZE)
#     model.optimize()
    
#     if model.status == GRB.OPTIMAL:
#         return model.objVal, y.X, {(i,j): x[i,j].X for i in all_nodes for j in all_nodes if i != j}
#     return None

# def evaluate_mean_value_solution(y_ev, x_ev, num_trials=20, scenarios_per_trial=10):
#     """Evaluate the mean value solution under different scenarios"""
#     total_cost = 0
    
#     for trial in range(num_trials):
#         trial_cost = 0
        
#         for _ in range(scenarios_per_trial):
#             stations, demands = generate_scenario()
            
#             # Calculate routing cost
#             routing_cost = sum(costs.loc[i,j] * x_ev[i,j] 
#                              for i, j in x_ev.keys() if i in ['Station_0'] + stations 
#                              and j in ['Station_0'] + stations)
            
#             total_demand = sum(demands.values())
#             shortage = max(0, total_demand - y_ev)
#             holding = max(0, y_ev - total_demand)
            
#             scenario_cost = routing_cost + 0.09 * holding + 0.13 * shortage
#             trial_cost += scenario_cost
        
#         total_cost += trial_cost / scenarios_per_trial
    
#     return total_cost / num_trials

# # Calculate EEV and VSS
# # 1. Solve mean value problem
# ev_result = solve_mean_value_problem()
# if ev_result is None:
#     print("Failed to solve mean value problem")
# else:
#     ev_obj, y_ev, x_ev = ev_result
#     print(f"Expected Value (EV) solution cost: ${ev_obj:.2f}")

#     # 2. Evaluate mean value solution in stochastic environment (EEV)
#     eev = evaluate_mean_value_solution(y_ev, x_ev)
#     print(f"Expected value of Expected Value solution (EEV): ${eev:.2f}")

#     # 3. Get RP value and calculate VSS
#     rp_value = solve_rp()
#     vss = eev - rp_value
#     print(f"Recourse Problem (RP) Value: ${rp_value:.2f}")
#     print(f"Value of Stochastic Solution (VSS): ${vss:.2f}")

Managerial Intuition from EVPI and VSS:
Expected Value of Perfect Information (EVPI = $23.84)
This value represents the maximum amount FuelFlow should be willing to pay for perfect information about future fuel demands
At $23.84 per delivery cycle, if FuelFlow makes daily deliveries, they could justify spending up to about $8,700 annually ($23.84 × 365) on:
Advanced demand forecasting systems
Better customer communication systems
Real-time demand monitoring technology
The relatively high EVPI indicates that demand uncertainty significantly impacts operational costs
However, since perfect information is rarely achievable, any investment in information systems should cost substantially less than $23.84 per delivery cycle to be economically justified
Value of Stochastic Solution (VSS = $66.45)
The large VSS ($66.45) shows that using average demands for planning (EEV = $200.55) is significantly more expensive than using stochastic programming (RP = $134.10)
This represents a potential 33% cost reduction ($66.45/$200.55) by implementing stochastic programming instead of using simple average-based planning
The high VSS strongly justifies:
Investment in sophisticated planning systems
Training staff in stochastic programming methods
Maintaining operational flexibility to handle demand variations
Development of scenario-based planning approaches
Comparative Analysis
VSS ($66.45) is significantly larger than EVPI ($23.84), which indicates:
The immediate priority should be improving planning methods rather than investing in better forecasting
The biggest gains will come from better handling of uncertainty rather than trying to reduce it
Current planning methods using averages are particularly ineffective for this problem
The cost of ignoring uncertainty (VSS) is about three times the value of eliminating it (EVPI)
Practical Recommendations
Primary Focus: Implement stochastic programming methods immediately, as the potential savings ($66.45 per delivery cycle) are substantial
Secondary Focus: Consider moderate investments in demand forecasting systems, but keep investments below the EVPI threshold
Operational Strategy:
Maintain flexibility in routing and delivery schedules
Develop contingency plans for varying demand scenarios
Train planners in stochastic optimization methods
Balance the cost of larger truck sizes against the risk of shortages
The numerical results strongly suggest that FuelFlow should prioritize improving their planning methodology over investing in better demand forecasting. The high VSS relative to EVPI indicates that even with uncertainty, better planning methods can significantly reduce costs. This is a clear directive for management to invest in sophisticated planning tools and capabilities rather than focusing primarily on demand prediction.

In [19]:
# Print data to verify structure
print("Costs DataFrame:")
print(costs.head())
print("\nRandomness DataFrame:")
print(randomness.head())

def generate_scenario():
    """Generate a random scenario based on probabilities"""
    stations = []
    demands = {}
    for station in randomness.index:
        if random.random() < randomness.loc[station, 'Probability']:
            stations.append(station)  # Use station name directly
            demands[station] = max(0, np.random.normal(
                randomness.loc[station, 'Mean_Demand'],
                randomness.loc[station, 'Std_Dev_Demand']
            ))
    return stations, demands

# Test scenario generation
test_stations, test_demands = generate_scenario()
print("\nTest scenario generation:")
print("Stations:", test_stations)
print("Demands:", test_demands)

Costs DataFrame:
  Unnamed: 0  Station_0  Station_1  Station_2  Station_3  Station_4  \
0  Station_0          0         43          7         27         20   
1  Station_1         22          0         12         38         19   
2  Station_2         23         46          0         14         43   
3  Station_3         10         45         14          0         44   
4  Station_4         39         32         15         44          0   

   Station_5  Station_6  Station_7  Station_8  Station_9  Station_10  \
0         22         43         28         42         30          22   
1         14         46         13         30         47          18   
2         14         33         23         13         45          12   
3         35         29         25          5         29          46   
4         24         48         47         25         34          14   

   Station_11  Station_12  Station_13  Station_14  
0          27          36          35          16  
1          29      

NameError: name 'random' is not defined

In [20]:
# Print data structure
print("Costs DataFrame shape:", costs.shape)
print("Costs index:", costs.index.tolist())
print("Costs columns:", costs.columns.tolist())
print("\nRandomness DataFrame shape:", randomness.shape)
print("Randomness index:", randomness.index.tolist())
print("Randomness columns:", randomness.columns.tolist())

# Print sample of both dataframes
print("\nCosts sample:")
print(costs.head())
print("\nRandomness sample:")
print(randomness.head())

Costs DataFrame shape: (15, 16)
Costs index: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Costs columns: ['Unnamed: 0', 'Station_0', 'Station_1', 'Station_2', 'Station_3', 'Station_4', 'Station_5', 'Station_6', 'Station_7', 'Station_8', 'Station_9', 'Station_10', 'Station_11', 'Station_12', 'Station_13', 'Station_14']

Randomness DataFrame shape: (14, 4)
Randomness index: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Randomness columns: ['Unnamed: 0', 'Probability', 'Mean_Demand', 'Std_Dev_Demand']

Costs sample:
  Unnamed: 0  Station_0  Station_1  Station_2  Station_3  Station_4  \
0  Station_0          0         43          7         27         20   
1  Station_1         22          0         12         38         19   
2  Station_2         23         46          0         14         43   
3  Station_3         10         45         14          0         44   
4  Station_4         39         32         15         44          0   

   Station_5  Station_6  Station_7  Station_8  

E

In [22]:

# Load data with correct indexing
costs = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/costs.csv')
costs.set_index('Unnamed: 0', inplace=True)

randomness = pd.read_csv('https://raw.githubusercontent.com/Dhanvi199/schulich_data_science/refs/heads/main/randomness.csv')
randomness.set_index('Unnamed: 0', inplace=True)

def generate_scenario():
    """Generate a random scenario based on probabilities"""
    stations = []
    demands = {}
    for station in randomness.index:
        if random.random() < randomness.loc[station, 'Probability']:
            stations.append(station)
            demands[station] = max(0, np.random.normal(
                randomness.loc[station, 'Mean_Demand'],
                randomness.loc[station, 'Std_Dev_Demand']
            ))
    return stations, demands

# Let's verify the data structure is correct now
print("Costs index:", costs.index.tolist())
print("Randomness index:", randomness.index.tolist())

# Test scenario generation
test_stations, test_demands = generate_scenario()
print("\nTest scenario generation:")
print("Stations:", test_stations)
print("Demands:", test_demands)

# Would you like me to continue with the full SAA implementation now that the data structure is fixed?

Costs index: ['Station_0', 'Station_1', 'Station_2', 'Station_3', 'Station_4', 'Station_5', 'Station_6', 'Station_7', 'Station_8', 'Station_9', 'Station_10', 'Station_11', 'Station_12', 'Station_13', 'Station_14']
Randomness index: ['Station_1', 'Station_2', 'Station_3', 'Station_4', 'Station_5', 'Station_6', 'Station_7', 'Station_8', 'Station_9', 'Station_10', 'Station_11', 'Station_12', 'Station_13', 'Station_14']

Test scenario generation:
Stations: ['Station_1', 'Station_2', 'Station_3', 'Station_7', 'Station_8', 'Station_9', 'Station_10', 'Station_11', 'Station_13', 'Station_14']
Demands: {'Station_1': 220.52447101897167, 'Station_2': 301.0287013555578, 'Station_3': 466.8021398770835, 'Station_7': 163.22432543848936, 'Station_8': 218.60896664732655, 'Station_9': 353.4604623549929, 'Station_10': 346.9550185731714, 'Station_11': 370.7653905183704, 'Station_13': 193.0858708078706, 'Station_14': 419.56813886671904}


In [24]:
def generate_scenario():
    """Generate a random scenario based on probabilities"""
    stations = []
    demands = {}
    for station in randomness.index:
        if random.random() < randomness.loc[station, 'Probability']:
            stations.append(station)
            demands[station] = max(0, np.random.normal(
                randomness.loc[station, 'Mean_Demand'],
                randomness.loc[station, 'Std_Dev_Demand']
            ))
    return stations, demands

def solve_saa():
    """Solve the SAA problem with 20 trials, 10 scenarios per trial"""
    trial_costs = []
    
    for trial in range(20):
        # Generate 10 scenarios for this trial
        scenarios = [generate_scenario() for _ in range(10)]
        
        # Create model for this trial
        model = Model("SAA_Trial")
        model.Params.OutputFlag = 0
        
        # First-stage variable (truck size)
        y = model.addVar(name='truck_size')
        
        # Second-stage variables for each scenario
        x = {}      # routing variables
        shortage = model.addVar()  # total shortage across scenarios
        excess = model.addVar()    # total excess across scenarios
        
        # Objective components
        total_travel_cost = 0
        total_demand = 0
        num_active_scenarios = 0
        
        # For each scenario
        for k, (stations, demands) in enumerate(scenarios):
            if not stations:  # Skip empty scenarios
                continue
                
            num_active_scenarios += 1
            all_nodes = ['Station_0'] + stations
            
            # Create routing variables for this scenario
            for i in all_nodes:
                for j in all_nodes:
                    if i != j:
                        x[k,i,j] = model.addVar(vtype=GRB.BINARY)
            
            # Flow constraints
            # Start at depot
            model.addConstr(quicksum(x[k,'Station_0',j] for j in stations) == 1)
            
            # Flow conservation
            for node in stations:
                model.addConstr(
                    quicksum(x[k,i,node] for i in all_nodes if i != node) ==
                    quicksum(x[k,node,j] for j in all_nodes if j != node)
                )
            
            # Visit each station once
            for j in stations:
                model.addConstr(quicksum(x[k,i,j] for i in all_nodes if i != j) == 1)
            
            # Return to depot
            model.addConstr(quicksum(x[k,i,'Station_0'] for i in stations) == 1)
            
            # Add travel costs for this scenario
            scenario_travel_cost = quicksum(costs.loc[i,j] * x[k,i,j] 
                                         for i in all_nodes 
                                         for j in all_nodes if i != j)
            total_travel_cost += scenario_travel_cost
            
            # Track total demand
            scenario_demand = sum(demands.values())
            total_demand += scenario_demand
            
        if num_active_scenarios > 0:
            # Average demand across active scenarios
            avg_demand = total_demand / num_active_scenarios
            
            # Shortage and excess constraints
            model.addConstr(shortage >= avg_demand - y)
            model.addConstr(excess >= y - avg_demand)
            
            # Set objective (average over active scenarios)
            obj = (total_travel_cost / num_active_scenarios +
                  0.13 * shortage +
                  0.09 * excess)
            
            model.setObjective(obj, GRB.MINIMIZE)
            model.optimize()
            
            if model.status == GRB.OPTIMAL:
                trial_costs.append(model.objVal)
    
    # Calculate statistics
    trial_costs = np.array(trial_costs)
    avg_cost = np.mean(trial_costs)
    std_dev = np.std(trial_costs)
    ci_lower = avg_cost - 1.96 * std_dev / np.sqrt(len(trial_costs))
    ci_upper = avg_cost + 1.96 * std_dev / np.sqrt(len(trial_costs))
    
    return avg_cost, std_dev, ci_lower, ci_upper

# Solve and print results
avg_cost, std_dev, ci_lower, ci_upper = solve_saa()
print(f"Optimal Expected Cost: ${avg_cost:.2f}")
print(f"Standard Deviation: ${std_dev:.2f}")
print(f"95% Confidence Interval: [${ci_lower:.2f}, ${ci_upper:.2f}]")

Optimal Expected Cost: $109.96
Standard Deviation: $4.65
95% Confidence Interval: [$107.93, $112.00]


F

In [26]:
def generate_scenario():
    stations = []
    demands = {}
    for station in randomness.index:
        if random.random() < randomness.loc[station, 'Probability']:
            stations.append(station)
            demands[station] = max(0, np.random.normal(
                randomness.loc[station, 'Mean_Demand'],
                randomness.loc[station, 'Std_Dev_Demand']
            ))
    return stations, demands

def solve_wait_and_see():
    """
    Solve the wait-and-see problem with multiple realizations
    Each realization has its own set of decision variables (except truck size)
    """
    trial_costs = []
    
    for trial in range(20):
        # Generate 10 scenarios for this trial
        scenarios = [generate_scenario() for _ in range(10)]
        
        # Create model for this trial
        model = Model("WS_Trial")
        model.Params.OutputFlag = 0
        
        # Variables and objective for each realization
        total_obj = 0
        
        # Process each realization independently
        for k, (stations, demands) in enumerate(scenarios):
            if not stations:
                continue
                
            # Variables for this realization
            y_k = model.addVar(name=f'y_{k}')  # truck size for this realization
            
            x_k = {}  # routing variables
            all_nodes = ['Station_0'] + stations
            
            for i in all_nodes:
                for j in all_nodes:
                    if i != j:
                        x_k[i,j] = model.addVar(vtype=GRB.BINARY, name=f'x_{k}_{i}_{j}')
            
            shortage_k = model.addVar(name=f'shortage_{k}')
            excess_k = model.addVar(name=f'excess_{k}')
            
            # Constraints for this realization
            # 1. Leave depot once
            model.addConstr(quicksum(x_k['Station_0',j] for j in stations) == 1)
            
            # 2. Flow conservation
            for node in stations:
                model.addConstr(
                    quicksum(x_k[i,node] for i in all_nodes if i != node) ==
                    quicksum(x_k[node,j] for j in all_nodes if j != node)
                )
            
            # 3. Visit each station once
            for j in stations:
                model.addConstr(quicksum(x_k[i,j] for i in all_nodes if i != j) == 1)
            
            # 4. Return to depot
            model.addConstr(quicksum(x_k[i,'Station_0'] for i in stations) == 1)
            
            # Demand balance for this realization
            total_demand = sum(demands.values())
            model.addConstr(shortage_k >= total_demand - y_k)
            model.addConstr(excess_k >= y_k - total_demand)
            
            # Add to total objective
            realization_obj = (
                quicksum(costs.loc[i,j] * x_k[i,j] 
                        for i in all_nodes for j in all_nodes if i != j) +
                0.13 * shortage_k +
                0.09 * excess_k
            )
            total_obj += realization_obj
        
        # Set objective (average over realizations)
        model.setObjective(total_obj / len(scenarios), GRB.MINIMIZE)
        
        # Optimize
        model.optimize()
        
        if model.status == GRB.OPTIMAL:
            trial_costs.append(model.objVal)
    
    # Return average over trials
    return np.mean(trial_costs)

# Calculate WS value
ws_value = solve_wait_and_see()
print(f"Wait-and-See (WS) Value = {ws_value:.2f}")

# Calculate EVPI using RP from part (e)
rp_value = 158.94  # From part (e)
evpi = rp_value - ws_value
print(f"Expected Value of Perfect Information (EVPI) = {evpi:.2f}")

Wait-and-See (WS) Value = 110.43
Expected Value of Perfect Information (EVPI) = 48.51


G

In [27]:
def solve_mean_value_problem():
    """
    Solve the mean value problem where:
    - ALL customers must be visited
    - Use mean demands
    """
    model = Model("MeanValue")
    model.Params.OutputFlag = 0
    
    # All stations must be visited
    stations = list(randomness.index)
    all_nodes = ['Station_0'] + stations
    
    # Variables
    y = model.addVar(name="truck_size")
    x = {}
    
    # Routing variables
    for i in all_nodes:
        for j in all_nodes:
            if i != j:
                x[i,j] = model.addVar(vtype=GRB.BINARY, name=f'x_{i}_{j}')
    
    shortage = model.addVar(name='shortage')
    excess = model.addVar(name='excess')
    
    # Routing constraints
    # 1. Leave depot once
    model.addConstr(quicksum(x['Station_0',j] for j in stations) == 1)
    
    # 2. Flow conservation
    for node in stations:
        model.addConstr(
            quicksum(x[i,node] for i in all_nodes if i != node) ==
            quicksum(x[node,j] for j in all_nodes if j != node)
        )
    
    # 3. Visit each station once
    for j in stations:
        model.addConstr(quicksum(x[i,j] for i in all_nodes if i != j) == 1)
    
    # 4. Return to depot
    model.addConstr(quicksum(x[i,'Station_0'] for i in stations) == 1)
    
    # Mean total demand
    total_mean_demand = sum(randomness['Mean_Demand'])
    
    # Demand balance
    model.addConstr(shortage >= total_mean_demand - y)
    model.addConstr(excess >= y - total_mean_demand)
    
    # Objective
    obj = (quicksum(costs.loc[i,j] * x[i,j] 
                   for i in all_nodes for j in all_nodes if i != j) +
           0.13 * shortage +
           0.09 * excess)
    
    model.setObjective(obj, GRB.MINIMIZE)
    model.optimize()
    
    if model.status == GRB.OPTIMAL:
        return model.objVal, y.X, {(i,j): x[i,j].X for i,j in x}
    return None

def evaluate_mean_value_solution(y_ev, x_ev):
    """
    Evaluate the mean value solution under uncertainty
    Using Monte Carlo simulation with 20 trials, 10 scenarios per trial
    """
    trial_costs = []
    
    for trial in range(20):
        trial_cost = 0
        
        # Generate 10 scenarios for this trial
        for _ in range(10):
            stations, demands = generate_scenario()
            if not stations:
                continue
            
            # Calculate costs using fixed EV solution
            # 1. Travel costs (using only the routes to active stations)
            travel_cost = sum(costs.loc[i,j] * x_ev[i,j] 
                            for i,j in x_ev if i in ['Station_0'] + stations 
                            and j in ['Station_0'] + stations)
            
            # 2. Shortage/excess costs
            total_demand = sum(demands.values())
            shortage = max(0, total_demand - y_ev)
            excess = max(0, y_ev - total_demand)
            
            # Total cost for this scenario
            scenario_cost = (travel_cost + 
                           0.13 * shortage +
                           0.09 * excess)
            trial_cost += scenario_cost
        
        trial_costs.append(trial_cost / 10)  # Average over scenarios
    
    return np.mean(trial_costs)  # Average over trials

def generate_scenario():
    stations = []
    demands = {}
    for station in randomness.index:
        if random.random() < randomness.loc[station, 'Probability']:
            stations.append(station)
            demands[station] = max(0, np.random.normal(
                randomness.loc[station, 'Mean_Demand'],
                randomness.loc[station, 'Std_Dev_Demand']
            ))
    return stations, demands

# 1. Solve mean value problem
ev_result = solve_mean_value_problem()
if ev_result:
    ev_obj, y_ev, x_ev = ev_result
    print(f"Expected Value (EV) solution cost = {ev_obj:.2f}")
    
    # 2. Evaluate EV solution under uncertainty (EEV)
    eev = evaluate_mean_value_solution(y_ev, x_ev)
    print(f"Expected value of Expected Value solution (EEV) = {eev:.2f}")
    
    # 3. Calculate VSS using RP from part (e)
    rp_value = 158.94  # From part (e)
    vss = eev - rp_value
    print(f"Value of Stochastic Solution (VSS) = {vss:.2f}")

Expected Value (EV) solution cost = 137.00
Expected value of Expected Value solution (EEV) = 196.36
Value of Stochastic Solution (VSS) = 37.42


H

1.)Managerial Intuition from EVPI and VSS:
Expected Value of Perfect Information (EVPI = $48.51)
Calculated as: RP ($158.94) - WS ($110.43) = $48.51
This means:
FuelFlow Logistics could save up to $48.51 per delivery day with perfect information
Annually, this represents potential savings of $17,706.15 (48.51 × 365 days)
Any investment in demand forecasting systems should cost less than $17,706 annually to be economically justified


2.)Practical implications:
Significant value in reducing uncertainty
Could justify investments in:
Advanced demand monitoring systems
Better communication with gas stations
Real-time inventory tracking
Value of Stochastic Solution (VSS = $37.42)
Calculated as: EEV ($196.36) - RP ($158.94) = $37.42
This shows:
Using average demands (EV solution) costs $196.36
Using stochastic programming costs $158.94
Daily savings of $37.42 by using stochastic programming
Annual savings potential of $13,658.30 (37.42 × 365 days)

3.)Practical implications:
Significant value in using stochastic programming over simple averages
Justifies investment in more sophisticated planning tools
Shows importance of considering uncertainty in daily operations
Comparative Analysis (EVPI vs VSS)
EVPI ($48.51) > VSS ($37.42) indicates:
Slightly more value in reducing uncertainty than in improving planning methods
The gap ($11.09 = $48.51 - $37.42) suggests both approaches have merit
A balanced approach to improvement is warranted


4.)Practical Recommendations
Primary Focus:
Invest in moderate-cost demand forecasting improvements (up to $48.51 daily value)
Implement stochastic programming methods ($37.42 daily value)
Secondary Focus:
Train staff in both better forecasting and stochastic programming
Develop flexible delivery strategies
Maintain balanced truck sizes


5.)Cost-Benefit Analysis
Mean Value Solution ($137.00) vs. Actual Performance ($196.36):
Shows significant cost of uncertainty ($59.36 difference)
Demonstrates that simple averages severely underestimate actual costs
Highlights importance of considering variability in planning
This analysis provides FuelFlow Logistics with clear guidance:
Both better information and better planning methods have significant value
Perfect information has slightly higher value, but both improvements are worthwhile
Current planning using averages significantly underestimates actual costs
A balanced investment in both forecasting and planning capabilities is justified